# Main Notebook: Bayesian Network Sampling and HPXML Mapping

This notebook performs the following tasks:<br>
[**1. Load the Bayesian Network**](#1.-Load-the-Bayesian-Network): Load the Bayesian Network from a `.XDSL` file. <br>
[**2. Execute Sampling**](2.-Execute-Sampling): Generate samples based on the Bayesian Network. <br>
[**3. Add Additional Variables**](#3.-Add-Additional-Variables): Add variables that are not part of the Bayesian Network. <br>
[**4. Map to HPXML Arguments**](#4.-Map-to-HPXML-Arguments): Map the sampled data to HPXML arguments. <br>
[**5. Save Results**](#5.-Save-Results): Save the results to CSV files for further processing. <br>

In [1]:
# importation of packages
import os
import sys
import pandas as pd
import dtale

current_dir = os.getcwd()
PROJECT_DIR = os.path.abspath(current_dir+ "/../")
sys.path.append(PROJECT_DIR)

from src.utils.sampler.Sampler import Sampler, BuildstockBatchArguments, MapHPXML

## **1. Load the Bayesian Network**

In [2]:
# Chargement du BN
InsClsSampler = Sampler()
path = PROJECT_DIR+"/data/processed/bayesian_network/BN_EUEMr.XDSL"
InsClsSampler.Load_BN(path)

## **2. Execute Sampling**

In [3]:
Nombre_de_Samples = 200
Evidence = {}
#Evidence = {"Type_Logement": "Maison en rangee"}
#            "Nombre_Pieces": "1"}#{"Mode_Occupation": "Proprietaire"}

# Exécution de l'échantillonnage
df1 = InsClsSampler.do_Sampling(Nombre_de_Samples, evs = Evidence)

## **3. Add Additional Variables**

In [4]:
lst_dct_args = df1.to_dict(orient='records')
# Affiche les échantillons - Avant enregistrement
#s.getBNStructure()
#print(s.lst_NOEUD, s.LIST_Dict)

#Ajout de varaible hors BN
Bba = BuildstockBatchArguments()
lst_dct_args2 = Bba.sampling( lst_dct_args)

lst_dct_args = [ d2 | d1 for d1, d2 in zip(lst_dct_args, lst_dct_args2)]#lst_dct_args prioritaire

## **4. Map to HPXML Arguments**

In [5]:
# Correspondance HPXML
MapSample = MapHPXML()
lst_dct_HPXML = MapSample.run(lst_dct_args)

#Affichage et enregistrement
print("Nombre d'attributs HPXML: ", len(lst_dct_HPXML[0].keys()))
#display(df1)

dfargs = pd.DataFrame(lst_dct_args)
dfHPXML = pd.DataFrame(lst_dct_HPXML)
#concat dataframe
dfAll = pd.concat([dfargs, dfHPXML], axis=1)
display(dfAll)

Nombre d'attributs HPXML:  134


,Geometry Stories,Geometry Building Number Units,Geometry Building Horizontal Location,Geometry Building Level,Geometry Foundation Type,Windows,Insulation Wall,Insulation Ceiling,Insulation Foundation Wall,Insulation Floor,...,overhangs_front_distance_to_top_of_window,misc_plug_loads_vehicle_annual_kwh,misc_plug_loads_vehicle_usage_multiplier,permanent_spa_pump_annual_kwh,permanent_spa_heater_annual_kwh,heat_pump_is_ducted,heat_pump_cooling_compressor_type,heat_pump_compressor_lockout_temp,heat_pump_heating_capacity_retention_fraction,heat_pump_heating_capacity_retention_temp
0,12,116,Left,Bottom,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R24.5,QC_R41,"QC_Wall-R17, interior",None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Low-E, L-Gain",QC_WoodStud-R14,QC_R24.6,"QC_Wall-R12, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9,326,Middle,Bottom,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R18.9,QC_R29.8,"QC_Wall-R17.1, interior",None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,326,Middle,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R12.2,QC_R12.5,"QC_Wall-R10.1, interior",None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Non-metal, Air",QC_WoodStud-R24.5,QC_R41,"QC_Wall-R17, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Non-metal, Air",QC_WoodStud-R17.2,QC_R27.3,"QC_Wall-R14.4, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,2,2,Not Applicable,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R12.2,QC_R12.5,"QC_Wall-R10.1, interior",None,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Metal, Air",QC_WoodStud-R17.2,QC_R27.3,"QC_Wall-R14.4, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,3,3,Not Applicable,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R18.9,QC_R29.8,"QC_Wall-R17.1, interior",None,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## **5. Save Results**

In [6]:
dfargs.to_csv(PROJECT_DIR+"/data/output/building-input.csv", index=False)
dfHPXML.to_csv(PROJECT_DIR+"/data/output/building-mapping.csv", index=False)
dfAll.to_csv(PROJECT_DIR+"/data/output/building-test.csv", index=False)

##dfHPXML.to_csv("N://Mes Documents//Projets LTE//Projet archQc//code//LTE-OPENSTUDIO-CLI//V7 Openstudio 3.7//LTE-OpenStudioCLI-Test//"+"building-mapping.csv", index=False)
#dfAll.to_csv("N://Mes Documents//Projets LTE//Projet archQc//code//LTE-OPENSTUDIO-CLI//V7 Openstudio 3.7//LTE-OpenStudioCLI-Test//"+"building-test.csv", index=False)

display(dfargs)
display(dfHPXML)
display(dfAll)


,Geometry Stories,Geometry Building Number Units,Geometry Building Horizontal Location,Geometry Building Level,Geometry Foundation Type,Windows,Insulation Wall,Insulation Ceiling,Insulation Foundation Wall,Insulation Floor,...,Eclairage_LED,Type_Batiment,An_Construction,Infiltration,An_ConstructionCode,Source_Energie_Chauf,Cuisiniere_Energie,Chauffage_Logement,Climatisation,ChaufEau_ChaufType
0,12,116,Left,Bottom,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R24.5,QC_R41,"QC_Wall-R17, interior",None,...,1 à 24 %,Collective,[2010 - 2020),Option=5 ACH50,>= 2013,Electricite,Electrique,Plinthes électriques,Murale,Electrique
1,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Low-E, L-Gain",QC_WoodStud-R14,QC_R24.6,"QC_Wall-R12, interior",Uninsulated,...,1 à 24 %,Maison,[1950 - 1960),Option=7 ACH50,[1946 - 1971),Electricite,Electrique,Système central à air chaud,Centrale,Electrique
2,9,326,Middle,Bottom,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R18.9,QC_R29.8,"QC_Wall-R17.1, interior",None,...,Plus de 50 %,Collective,[1990 - 2000),Option=4 ACH50,[1986 - 2013),Electricite,Electrique,Plinthes électriques,Aucune,Electrique
3,8,326,Middle,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R12.2,QC_R12.5,"QC_Wall-R10.1, interior",None,...,0%,Collective,< 1950,Option=6 ACH50,< 1946,Electricite,Electrique,Plinthes électriques,Aucune,Electrique
4,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Non-metal, Air",QC_WoodStud-R24.5,QC_R41,"QC_Wall-R17, interior",Uninsulated,...,1 à 24 %,Maison,[2010 - 2020),Option=1 ACH50,[1986 - 2013),Bois,Electrique,Fournaise ou poêle à bois et Plinthes électriques,"Fenêtre, mobile, portable",Electrique
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Non-metal, Air",QC_WoodStud-R17.2,QC_R27.3,"QC_Wall-R14.4, interior",Uninsulated,...,Plus de 50 %,Maison,[1970 - 1980),Option=7 ACH50,[1971 - 1986),Electricite,Electrique,Système central à air chaud,Centrale,Electrique
196,2,2,Not Applicable,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R12.2,QC_R12.5,"QC_Wall-R10.1, interior",None,...,1 à 24 %,Plex,< 1950,Option=6 ACH50,< 1946,Electricite,Electrique,Plinthes électriques,"Fenêtre, mobile, portable",Electrique
197,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Metal, Air",QC_WoodStud-R17.2,QC_R27.3,"QC_Wall-R14.4, interior",Uninsulated,...,Plus de 50 %,Maison,[1970 - 1980),Option=5 ACH50,[1971 - 1986),Electricite,Electrique,Plinthes électriques,Aucune,Electrique
198,3,3,Not Applicable,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R18.9,QC_R29.8,"QC_Wall-R17.1, interior",None,...,0%,Plex,[1980 - 1990),Option=1 ACH50,[1986 - 2013),Electricite,Electrique,Plinthes électriques,Aucune,Electrique


,weather_station_epw_filepath,simulation_control_daylight_saving_enabled,site_time_zone_utc_offset,geometry_building_num_units,geometry_unit_type,geometry_average_ceiling_height,geometry_unit_aspect_ratio,geometry_unit_cfa,year_built,geometry_unit_num_bedrooms,...,overhangs_front_distance_to_top_of_window,misc_plug_loads_vehicle_annual_kwh,misc_plug_loads_vehicle_usage_multiplier,permanent_spa_pump_annual_kwh,permanent_spa_heater_annual_kwh,heat_pump_is_ducted,heat_pump_cooling_compressor_type,heat_pump_compressor_lockout_temp,heat_pump_heating_capacity_retention_fraction,heat_pump_heating_capacity_retention_temp
0,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,116,apartment unit,8,0.5556,750,2015,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,1,single-family detached,8,1.8000,2179,1955,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,326,apartment unit,8,0.5556,750,1995,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,326,apartment unit,8,0.5556,1138,1940,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020s_CAN_QC_Saguenay-Bagotville.AP-CFB.Bagotv...,True,-5,1,single-family detached,8,1.8000,2179,2015,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,1,single-family detached,8,1.8000,1228,1975,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,2,apartment unit,8,0.5556,1138,1940,3,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,1,single-family detached,8,1.8000,1698,1975,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,2020s_CAN_QC_Montreal-McTavish.716120_CWEC2016...,True,-5,3,apartment unit,8,0.5556,3250,1985,3,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Geometry Stories,Geometry Building Number Units,Geometry Building Horizontal Location,Geometry Building Level,Geometry Foundation Type,Windows,Insulation Wall,Insulation Ceiling,Insulation Foundation Wall,Insulation Floor,...,overhangs_front_distance_to_top_of_window,misc_plug_loads_vehicle_annual_kwh,misc_plug_loads_vehicle_usage_multiplier,permanent_spa_pump_annual_kwh,permanent_spa_heater_annual_kwh,heat_pump_is_ducted,heat_pump_cooling_compressor_type,heat_pump_compressor_lockout_temp,heat_pump_heating_capacity_retention_fraction,heat_pump_heating_capacity_retention_temp
0,12,116,Left,Bottom,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R24.5,QC_R41,"QC_Wall-R17, interior",None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Low-E, L-Gain",QC_WoodStud-R14,QC_R24.6,"QC_Wall-R12, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9,326,Middle,Bottom,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R18.9,QC_R29.8,"QC_Wall-R17.1, interior",None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,326,Middle,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R12.2,QC_R12.5,"QC_Wall-R10.1, interior",None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Non-metal, Air",QC_WoodStud-R24.5,QC_R41,"QC_Wall-R17, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Non-metal, Air",QC_WoodStud-R17.2,QC_R27.3,"QC_Wall-R14.4, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
196,2,2,Not Applicable,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R12.2,QC_R12.5,"QC_Wall-R10.1, interior",None,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197,1,1,Not Applicable,Not Applicable,Heated Basement,"Double, Clear, Metal, Air",QC_WoodStud-R17.2,QC_R27.3,"QC_Wall-R14.4, interior",Uninsulated,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
198,3,3,Not Applicable,Top,Slab,"Double, Clear, Non-metal, Air",QC_WoodStud-R18.9,QC_R29.8,"QC_Wall-R17.1, interior",None,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
dtale.show(dfargs)

In [8]:
dtale.show(dfHPXML)